In [1]:
# optional but recommended
import shutil
shutil.rmtree("faiss_index", ignore_errors=True)

In [2]:
!pip install -q langchain langchain-community langchain-text-splitters faiss-cpu pypdf transformers

In [3]:
from google.colab import files

uploaded = files.upload()

Saving dummy.pdf to dummy (1).pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

file_name = list(uploaded.keys())[0]

loader = PyPDFLoader(file_name)
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 1


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 1


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector DB created")

/tmp/ipykernel_34734/3776192777.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_34734/3776192777.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created


In [7]:
retriever = vectorstore.as_retriever()

In [8]:
!pip install -U transformers

In [12]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    max_new_tokens=100
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCa

Model loaded successfully!


In [13]:
def ask_pdf(query):

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs[:3]])

    prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}
"""

    result = pipe(prompt)

    return result[0]["generated_text"]

In [14]:
print(ask_pdf("What is this document about?"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question based only on the context below.

Context:
Dummy PDF file

Question:
What is this document about?



In [15]:
print(ask_pdf("Summarize this document"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question based only on the context below.

Context:
Dummy PDF file

Question:
Summarize this document

